# Metrics Sweep

**Two experimental setups.** EmbeddingGemma outputs L2-normalized
vectors, so on the raw file all three metrics (L2, IP, cosine) are
ordinally equivalent — the sweep degenerates to "did the algorithm
find the same neighbours regardless of metric". We run two setups:

1. **Real embeddings, as-is** — normalized production data. The
   headline result here is: *on normalized data, metric choice barely
   matters; algorithm choice dominates.*
2. **Denormalized counterfactual** — same real vectors × random
   log-uniform magnitudes. Shows what the sweep would tell you if
   embeddings were not normalized (e.g. some vision or multimodal
   models). Not a proposal to actually run this in production.

Fallback to MockSource if `embeds.npy` isn't present.


## 1. Setup

In [2]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss

from indexing import MockSource, NumpyFileSource

# ---- knobs ----
EMBEDS_PATH = ROOT / "embeds.npy"

FALLBACK_N = 20_000
FALLBACK_DIM = 768
FALLBACK_HIDDEN = 50

N_QUERIES = 200
TOPK = 10
SEED = 0

print(f"faiss version: {faiss.__version__}")


faiss version: 1.14.3


## 2. Load real embeddings

Materialize the whole array once (mmap → RAM) — the metric sweep hits
it many times.


In [3]:
if EMBEDS_PATH.exists():
    src = NumpyFileSource(EMBEDS_PATH)
    raw = np.asarray(src._vectors[:], dtype=np.float32)
    print(f"real embeddings   : {EMBEDS_PATH}  N={raw.shape[0]:,}  dim={raw.shape[1]}")
else:
    src = MockSource(n=FALLBACK_N, dim=FALLBACK_DIM,
                     n_clusters_true=FALLBACK_HIDDEN, seed=0)
    raw = src._vectors.astype(np.float32)
    print(f"WARNING: {EMBEDS_PATH.name} not found — fallback MockSource "
          f"(N={raw.shape[0]:,}, dim={raw.shape[1]})")

N, DIM = raw.shape

# Sanity: how normalized are the raw vectors?
raw_norms = np.linalg.norm(raw, axis=1)
print(f"raw norms         : min={raw_norms.min():.3f}  "
      f"max={raw_norms.max():.3f}  mean={raw_norms.mean():.3f}")
if abs(raw_norms.mean() - 1.0) < 0.01:
    print("  -> looks L2-normalized (EmbeddingGemma default)")


raw norms         : min=1.000  max=1.000  mean=1.000
  -> looks L2-normalized (EmbeddingGemma default)


## 3. Two corpus variants + query set

**A. as-is** — real embeddings, forced to unit norm for safety.
**B. denormalized** — same embeddings scaled by log-uniform magnitudes.

Both use the same 200 held-out queries (same underlying vectors) with
their respective transforms applied.


In [4]:
def l2_normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


# Variant A: normalized (production).
corpus_A = l2_normalize(raw).astype(np.float32)

# Variant B: denormalized counterfactual.
rng = np.random.default_rng(0)
mags = np.exp(rng.uniform(np.log(0.5), np.log(2.0), size=(N, 1))).astype(np.float32)
corpus_B = corpus_A * mags

# Queries: sample once, apply variant transforms.
qry_idx = np.random.default_rng(123).choice(N, size=N_QUERIES, replace=False)
queries_A = corpus_A[qry_idx].copy()
queries_B = corpus_B[qry_idx].copy()

print(f"Variant A norms   : min={np.linalg.norm(corpus_A, axis=1).min():.3f}  "
      f"max={np.linalg.norm(corpus_A, axis=1).max():.3f}")
print(f"Variant B norms   : min={np.linalg.norm(corpus_B, axis=1).min():.3f}  "
      f"max={np.linalg.norm(corpus_B, axis=1).max():.3f}")


Variant A norms   : min=1.000  max=1.000
Variant B norms   : min=0.500  max=2.000


## 4. Ground truth per metric per variant

Six brute-force sweeps (3 metrics × 2 variants). Each stores its own
top-K, and every approximate index is evaluated against the GT that
matches both its metric and its variant.


In [5]:
def brute_gt(metric, corpus, queries, topk):
    if metric == "cosine":
        c = corpus / np.linalg.norm(corpus, axis=1, keepdims=True)
        q = queries / np.linalg.norm(queries, axis=1, keepdims=True)
        idx = faiss.IndexFlatIP(c.shape[1]); idx.add(c.astype(np.float32))
        q = q.astype(np.float32)
    elif metric == "ip":
        idx = faiss.IndexFlatIP(corpus.shape[1]); idx.add(corpus); q = queries
    elif metric == "l2":
        idx = faiss.IndexFlatL2(corpus.shape[1]); idx.add(corpus); q = queries
    else:
        raise ValueError(metric)
    _, gt = idx.search(q, topk)
    return gt


gts_A = {m: brute_gt(m, corpus_A, queries_A, TOPK) for m in ("l2", "ip", "cosine")}
gts_B = {m: brute_gt(m, corpus_B, queries_B, TOPK) for m in ("l2", "ip", "cosine")}


def overlap_matrix(gts):
    df = pd.DataFrame(index=list(gts), columns=list(gts), dtype=float)
    for a, ga in gts.items():
        for b, gb in gts.items():
            h = sum(len(set(ga[i]) & set(gb[i])) for i in range(N_QUERIES))
            df.loc[a, b] = round(h / (N_QUERIES * TOPK), 3)
    return df


print("Variant A (real, normalized) — GT overlap:")
overlap_A = overlap_matrix(gts_A); print(overlap_A)
print("\nVariant B (denormalized counterfactual) — GT overlap:")
overlap_B = overlap_matrix(gts_B); print(overlap_B)


Variant A (real, normalized) — GT overlap:
         l2   ip  cosine
l2      1.0  1.0     1.0
ip      1.0  1.0     1.0
cosine  1.0  1.0     1.0

Variant B (denormalized counterfactual) — GT overlap:
          l2     ip  cosine
l2      1.00  0.100   0.210
ip      0.10  1.000   0.336
cosine  0.21  0.336   1.000


### Reading the overlap matrices

If `overlap_A` is ≈ all 1.0, that's the expected sanity result — on
production-grade normalized embeddings all three metrics find the
same neighbours, so metric choice makes no difference. If it isn't,
the file wasn't strictly normalized.

`overlap_B` should be very different from `overlap_A`: L2 and IP now
disagree because magnitudes vary. This is the "why metrics matter"
demonstration.


## 5. Harness — one function for every (variant × algo × metric × config)

In [6]:
def bench(idx, method, variant, metric, queries, gt, param_configs, topk, build_s):
    rows = []
    for label, setter in param_configs:
        if setter is not None:
            setter(idx)
        _ = idx.search(queries[:1], topk)  # warmup
        t = time.perf_counter()
        _, labels = idx.search(queries, topk)
        latency_ms = (time.perf_counter() - t) / len(queries) * 1000
        hits = sum(
            len(set(labels[i].tolist()) & set(gt[i].tolist()))
            for i in range(len(labels))
        )
        recall = hits / (len(labels) * topk)
        rows.append({
            "variant": variant, "method": method, "metric": metric,
            "config": label, "recall@10": recall, "latency_ms": latency_ms,
            "build_s": round(build_s, 3),
        })
    return rows


def prepare_for_metric(metric, corpus, queries):
    if metric == "l2":
        return corpus, queries, faiss.METRIC_L2
    if metric == "ip":
        return corpus, queries, faiss.METRIC_INNER_PRODUCT
    if metric == "cosine":
        cnorm = corpus / np.linalg.norm(corpus, axis=1, keepdims=True)
        qnorm = queries / np.linalg.norm(queries, axis=1, keepdims=True)
        return cnorm.astype(np.float32), qnorm.astype(np.float32), faiss.METRIC_INNER_PRODUCT
    raise ValueError(metric)


all_results = []


## 6. IVF across (variants × metrics)

In [7]:
nlist = 128

for variant, corpus_v, queries_v, gts_v in [
    ("A_real",     corpus_A, queries_A, gts_A),
    ("B_denorm",   corpus_B, queries_B, gts_B),
]:
    for metric in ["l2", "ip", "cosine"]:
        c, q, fm = prepare_for_metric(metric, corpus_v, queries_v)
        quant = faiss.IndexFlatL2(DIM) if fm == faiss.METRIC_L2 else faiss.IndexFlatIP(DIM)
        idx = faiss.IndexIVFFlat(quant, DIM, nlist, fm)
        t = time.perf_counter()
        idx.train(c); idx.add(c)
        build = time.perf_counter() - t
        gt = gts_v[metric]
        configs = [(f"nprobe={n}",
                    (lambda n_: (lambda i: setattr(i, "nprobe", n_)))(n))
                   for n in [1, 2, 4, 8, 16, 32]]
        rows = bench(idx, "ivf", variant, metric, q, gt, configs, TOPK, build)
        all_results += rows
        print(f"  ivf  {variant:<10} + {metric:<7}  build={build:.2f}s  "
              f"best_recall={max(r['recall@10'] for r in rows):.3f}")


  ivf  A_real     + l2       build=0.11s  best_recall=0.898
  ivf  A_real     + ip       build=0.07s  best_recall=0.876
  ivf  A_real     + cosine   build=0.07s  best_recall=0.876
  ivf  B_denorm   + l2       build=0.10s  best_recall=1.000
  ivf  B_denorm   + ip       build=0.07s  best_recall=0.782
  ivf  B_denorm   + cosine   build=0.07s  best_recall=0.876


## 7. HNSW across (variants × metrics)

In [8]:
M_hnsw = 16

for variant, corpus_v, queries_v, gts_v in [
    ("A_real",   corpus_A, queries_A, gts_A),
    ("B_denorm", corpus_B, queries_B, gts_B),
]:
    for metric in ["l2", "ip", "cosine"]:
        c, q, fm = prepare_for_metric(metric, corpus_v, queries_v)
        idx = faiss.IndexHNSWFlat(DIM, M_hnsw, fm)
        idx.hnsw.efConstruction = 200
        t = time.perf_counter()
        idx.add(c)
        build = time.perf_counter() - t
        gt = gts_v[metric]
        configs = [(f"efSearch={ef}",
                    (lambda ef_: (lambda i: setattr(i.hnsw, "efSearch", ef_)))(ef))
                   for ef in [10, 20, 40, 80, 160, 320]]
        rows = bench(idx, "hnsw", variant, metric, q, gt, configs, TOPK, build)
        all_results += rows
        print(f"  hnsw {variant:<10} + {metric:<7}  build={build:.2f}s  "
              f"best_recall={max(r['recall@10'] for r in rows):.3f}")


  hnsw A_real     + l2       build=2.47s  best_recall=0.941
  hnsw A_real     + ip       build=2.63s  best_recall=0.939
  hnsw A_real     + cosine   build=2.88s  best_recall=0.945
  hnsw B_denorm   + l2       build=0.66s  best_recall=0.899
  hnsw B_denorm   + ip       build=1.50s  best_recall=0.949
  hnsw B_denorm   + cosine   build=2.46s  best_recall=0.936


## 8. NSG across (variants × metrics)

In [ ]:
R = 16

for variant, corpus_v, queries_v, gts_v in [
    ("A_real",   corpus_A, queries_A, gts_A),
    ("B_denorm", corpus_B, queries_B, gts_B),
]:
    for metric in ["l2", "ip", "cosine"]:
        c, q, fm = prepare_for_metric(metric, corpus_v, queries_v)
        idx = faiss.IndexNSGFlat(DIM, R, fm)
        t = time.perf_counter()
        idx.add(c)
        build = time.perf_counter() - t
        gt = gts_v[metric]
        configs = [(f"search_L={L}",
                    (lambda L_: (lambda i: setattr(i.nsg, "search_L", L_)))(L))
                   for L in [16, 32, 64, 128, 256]]
        rows = bench(idx, "nsg", variant, metric, q, gt, configs, TOPK, build)
        all_results += rows
        print(f"  nsg  {variant:<10} + {metric:<7}  build={build:.2f}s  "
              f"best_recall={max(r['recall@10'] for r in rows):.3f}")

df = pd.DataFrame(all_results)
df.to_csv(ROOT / "experiments" / "metrics_sweep_real.csv", index=False)
print(f"\nsaved {len(df)} rows to experiments/metrics_sweep_real.csv")


## 9. Pareto plots — one grid per variant

Rows = variants (A_real vs B_denorm), cols = metrics. Each subplot has
three curves (IVF, HNSW, NSG).


In [ ]:
styles = {
    "ivf":  {"marker": "o", "color": "tab:blue"},
    "hnsw": {"marker": "s", "color": "tab:purple"},
    "nsg":  {"marker": "^", "color": "tab:green"},
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for row_i, variant in enumerate(["A_real", "B_denorm"]):
    for col_j, metric in enumerate(["l2", "ip", "cosine"]):
        ax = axes[row_i, col_j]
        sub_vm = df[(df["variant"] == variant) & (df["metric"] == metric)]
        for method, sub in sub_vm.groupby("method"):
            sub = sub.sort_values("latency_ms")
            s = styles[method]
            ax.plot(sub["latency_ms"], sub["recall@10"],
                    marker=s["marker"], color=s["color"], label=method,
                    linewidth=1.5, markersize=8, alpha=0.9)
        ax.set_xscale("log")
        ax.grid(alpha=0.3)
        ax.set_title(f"{variant} × {metric}")
        if row_i == 1: ax.set_xlabel("latency (ms) — log")
        if col_j == 0: ax.set_ylabel("recall@10")
        if row_i == 0 and col_j == 0: ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()


## 10. Summary — best recall by variant × algo × metric

In [ ]:
summary = (
    df.groupby(["variant", "method", "metric"])
      .agg(
          best_recall=("recall@10", "max"),
          fastest_ms=("latency_ms", "min"),
          build_s=("build_s", "first"),
      )
      .round(3)
      .sort_index()
)
summary


## 11. Takeaways

**Variant A — real EmbeddingGemma vectors.**

If the vectors are strictly L2-normalized (mean norm ≈ 1.0), then L2 /
IP / cosine give ordinally equivalent nearest-neighbour rankings — the
GT overlap matrix should be all 1.0, and the three subplots in row 1
should look near-identical. This is the practical message:
**for production RAG on EmbeddingGemma, pick whatever FAISS metric
is fastest — cosine (via IP on normalized vectors) is usually the
right choice because the IP kernels are well-optimized.**

**Variant B — denormalized counterfactual.**

The B row demonstrates what happens with unnormalized data. L2 and IP
diverge (their GT overlap drops), and each algorithm's behaviour under
each metric now differs. This is the setup relevant if you ever adopt
a model that doesn't normalize (some vision encoders, MoCo variants,
some multimodal models).

**Algorithmic pattern (holds across both variants).**

- HNSW/NSG dominate the high-recall end; both are metric-agnostic and
  usually within a few percent of each other.
- IVF wins on build speed and memory but pays a small recall tax at
  the top end.
- On IVF, the L2 case is often marginally better than IP — because the
  k-means step is L2-native. This is where our custom SphericalKMeans
  (in `indexing/clustering.py`) closes the gap for cosine.

### Next steps

- Merge these findings with the FAISS benchmark and MRL ablation into
  a single "Level-3 report" plot showing all three axes: algorithm ×
  metric × MRL slice, all on real embeddings.
- On variant A the sweep is boring by design — consider dropping it
  from the report and keeping only variant B as an "if you ever go
  unnormalized" reference.
